# MineToday V129 — Lean Sparse×Dense Fully-Nested Top1 Utility Fusion

**Frozen experiment contract.** V129 implements the post-V128 audit decision without adding a new representation family: remove BM25 and all target-neighbor/template target-stat features; keep word/character TF-IDF, E5, Nomic, assessments/readiness, and anchor geometry; use 17 per-module LightGBM regressors; train a **fully nested** Ridge head directly on exact counterfactual Top1 ΔNDCG; and deploy a **fixed 5%** S0-only Top1 score-swap policy over the exact `0.66028` champion.

Primary validation is five-fold random-user CV with true second-level nesting. Seen/unseen template performance is mixed using the frozen test-side estimate `81.6% / 18.4%`. Exact-history, latest-skeleton, near-template, and contiguous-ID worlds are diagnostics rather than individual hard vetoes. There is no prospective LOCK, no `G_safe`, no benefit:harm gate, and no coverage search.

**Submission decision:** correctness failures always fail closed. Model-quality status is `GREEN`, `YELLOW`, `BORDERLINE`, or `RED`; only `GREEN/YELLOW` materializes the candidate as `submission_v129_primary.csv`. `candidate_v129_DIAGNOSTIC.csv` is always written for audit. Full run target: Kaggle **T4×2**. Set `V129_SMOKE=1` only for a runtime smoke test; smoke outputs are never submission-eligible.

In [1]:
# =========================
# 0. Runtime / dependencies / frozen configuration
# =========================
import os, sys, re, io, gc, json, ast, html, math, time, base64, gzip, hashlib, random, warnings, unicodedata, subprocess, platform
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict, Counter

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")
os.environ.setdefault("PYTHONHASHSEED", "20260907")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
warnings.filterwarnings("ignore")

# V47/V48 failed on dependency drift. Install only when the active runtime is incompatible.
def _ensure_runtime():
    import importlib.metadata as im
    need=[]
    try: im.version("lightgbm")
    except Exception: need.append("lightgbm==4.6.0")
    try:
        tv=im.version("transformers")
        if int(tv.split('.')[0]) >= 5: need += ["transformers==4.57.1", "sentence-transformers==5.1.1"]
    except Exception: need += ["transformers==4.57.1", "sentence-transformers==5.1.1"]
    try: im.version("sentence-transformers")
    except Exception:
        if not any(x.startswith("sentence-transformers") for x in need): need.append("sentence-transformers==5.1.1")
    try: im.version("einops")
    except Exception: need.append("einops>=0.8.0")
    if need:
        print("Installing compatibility dependencies before importing model libraries:", need)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])

_ensure_runtime()

import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from scipy.sparse import hstack, csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import lightgbm as lgb

SEED=20260907
random.seed(SEED); np.random.seed(SEED)

@dataclass(frozen=True)
class CFG:
    seed:int=SEED
    n_outer:int=5
    n_inner:int=3
    word_max_features:int=60000
    char_max_features:int=80000
    near_threshold:float=0.90
    lgb_estimators:int=160
    lgb_lr:float=0.03
    lgb_num_leaves:int=15
    lgb_max_depth:int=4
    lgb_min_child:int=80
    lgb_subsample:float=0.80
    lgb_colsample:float=0.75
    lgb_reg_alpha:float=0.5
    lgb_reg_lambda:float=8.0
    lgb_seeds:tuple=(20260907,20260917,20260927)
    action_alpha:float=10.0
    fixed_coverage:float=0.05
    diagnostic_coverages:tuple=(0.03,0.04,0.05,0.06)
    seen_weight:float=0.816
    green_gain:float=0.0008
    yellow_gain:float=0.0005
    catastrophic_stress:float=-0.001
    bootstraps:int=2000
    smoke:bool=bool(int(os.getenv("V129_SMOKE","0")))
    run_stress:bool=bool(int(os.getenv("V129_RUN_STRESS","1")))

cfg=CFG()
if cfg.smoke:
    cfg=CFG(n_outer=2,n_inner=2,word_max_features=8000,char_max_features=12000,lgb_estimators=30,
            lgb_seeds=(20260907,),bootstraps=100,smoke=True,run_stress=False)

OUT=Path("/kaggle/working/v129") if Path("/kaggle/working").exists() else Path("./v129")
OUT.mkdir(parents=True,exist_ok=True)
print("CFG",asdict(cfg))
print("numpy",np.__version__,"pandas",pd.__version__,"scipy",scipy.__version__,"sklearn",__import__('sklearn').__version__,"lightgbm",lgb.__version__)
try: print(subprocess.check_output(["nvidia-smi","-L"],text=True).strip())
except Exception: print("GPU not visible in this preflight; full V129 is intended for Kaggle T4x2.")


Installing compatibility dependencies before importing model libraries: ['transformers==4.57.1', 'sentence-transformers==5.1.1']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 626.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.6 MB/s eta 0:00:00
CFG {'seed': 20260907, 'n_outer': 5, 'n_inner': 3, 'word_max_features': 60000, 'char_max_features': 80000, 'near_threshold': 0.9, 'lgb_estimators': 160, 'lgb_lr': 0.03, 'lgb_num_leaves': 15, 'lgb_max_depth': 4, 'lgb_min_child': 80, 'lgb_subsample': 0.8, 'lgb_colsample': 0.75, 'lgb_reg_alpha': 0.5, 'lgb_reg_lambda': 8.0, 'lgb_seeds': (20260907, 20260917, 20260927), 'action_alpha': 10.0, 'fixed_coverage': 0.05, 'diagnostic_coverages': (0.03, 0.04, 0.05, 0.06), 'seen_weight': 0.816, 'green_gain': 0.0008, 'yellow_gain': 0.0005, 'catastrophic_stres

In [2]:
# =========================
# 1. Frozen champion/protection + organizer data loading
# =========================
CHAMPION_SHA256="443785fb534df9820ce57bfe450a2768881625bf7ae37aa849dd0d9f2afc957d"
CHAMPION_TOP5_SHA256="7c0d29a7309b7283a043b7d37ca5556ba7c0ca2270b0137a6e1e0878a69a5fe7"
CHAMPION_GZ_B64="""..."""
NON_S0_TEST_USERS=set(["U_4005", "U_4015", "U_4017", "U_4043", "U_4044", "U_4058", "U_4068", "U_4069", "U_4070", "U_4075", "U_4087", "U_4089", "U_4091", "U_4098", "U_4105", "U_4118", "U_4121", "U_4122", "U_4123", "U_4130", "U_4131", "U_4137", "U_4138", "U_4169", "U_4178", "U_4183", "U_4186", "U_4203", "U_4214", "U_4219", "U_4232", "U_4233", "U_4237", "U_4265", "U_4266", "U_4269", "U_4274", "U_4285", "U_4289", "U_4300", "U_4304", "U_4320", "U_4335", "U_4345", "U_4354", "U_4363", "U_4379", "U_4389", "U_4394", "U_4402", "U_4407", "U_4430", "U_4436", "U_4438", "U_4442", "U_4480", "U_4515", "U_4527", "U_4532", "U_4542", "U_4545", "U_4555", "U_4569", "U_4580", "U_4582", "U_4596", "U_4601", "U_4633", "U_4639", "U_4650", "U_4661", "U_4663", "U_4665", "U_4672", "U_4675", "U_4687", "U_4717", "U_4718", "U_4725", "U_4730", "U_4751", "U_4759", "U_4763", "U_4827", "U_4828", "U_4838", "U_4886", "U_4887", "U_4893", "U_4896", "U_4897", "U_4905", "U_4919", "U_4933", "U_4942", "U_4951", "U_4956", "U_4960", "U_4965", "U_4971", "U_4978", "U_4988", "U_4989"])
V25_PROTECTED_TEST_USERS={"U_4047","U_4953","U_4074","U_4176","U_4491","U_4567","U_4836","U_4929"}

CHAMPION_BYTES=gzip.decompress(base64.b64decode(CHAMPION_GZ_B64.encode()))
assert hashlib.sha256(CHAMPION_BYTES).hexdigest()==CHAMPION_SHA256
champion=pd.read_csv(io.BytesIO(CHAMPION_BYTES))

REQUIRED={"chat_history.csv","modules_catalog.csv","user_assessments.csv","train_relevance.csv","test.csv","sample_submission.csv"}
def locate_data():
    roots=[Path("/kaggle/input"),Path("."),Path("/mnt/data")]
    for root in roots:
        if not root.exists(): continue
        cands=[root]
        if root.name in {"input","data"} or root==Path("/kaggle/input"):
            cands += [p for p in root.rglob("*") if p.is_dir()]
        for d in cands:
            try: names={p.name for p in d.iterdir() if p.is_file()}
            except Exception: continue
            if REQUIRED.issubset(names): return d
    raise FileNotFoundError("MineToday organizer CSV directory not found under /kaggle/input.")

DATA=locate_data(); print("DATA",DATA)
chat=pd.read_csv(DATA/"chat_history.csv")
modules=pd.read_csv(DATA/"modules_catalog.csv")
assess=pd.read_csv(DATA/"user_assessments.csv")
train=pd.read_csv(DATA/"train_relevance.csv")
test=pd.read_csv(DATA/"test.csv")
sample=pd.read_csv(DATA/"sample_submission.csv")
MODULES=list(train.columns[1:]); M=len(MODULES); NTR=len(train); NTE=len(test); N=NTR+NTE
assert M==17 and NTR==4000 and NTE==1000
assert MODULES==list(sample.columns[1:])==list(champion.columns[1:])
assert champion.user_id.astype(str).tolist()==test.user_id.astype(str).tolist()
ALL_USERS=train.user_id.astype(str).tolist()+test.user_id.astype(str).tolist(); UID2I={u:i for i,u in enumerate(ALL_USERS)}
y=train[MODULES].to_numpy(np.float32)

# Frozen deployment geometry from direct V30/V31 ledgers: union=103 non-S0; exact S0 count=897.
assert len(NON_S0_TEST_USERS)==103
assert len(V25_PROTECTED_TEST_USERS)==8
TEST_ELIGIBLE=np.array([(u not in NON_S0_TEST_USERS) and (u not in V25_PROTECTED_TEST_USERS) for u in test.user_id.astype(str)],bool)
assert int(TEST_ELIGIBLE.sum())==889

DISCOUNT=1/np.log2(np.arange(2,7,dtype=np.float64))
def top5_idx(scores):
    s=np.asarray(scores); ix=np.argpartition(-s,4,axis=1)[:,:5]; loc=np.take_along_axis(s,ix,1)
    return np.take_along_axis(ix,np.argsort(-loc,axis=1,kind="stable"),1)
def top5_hash(scores): return hashlib.sha256(top5_idx(scores).astype(np.int16).tobytes()).hexdigest()
def per_user_ndcg(yy,ss):
    ix=top5_idx(ss); dcg=(np.take_along_axis(yy,ix,1)*DISCOUNT).sum(1)
    ideal=np.sort(yy,axis=1)[:,-5:][:,::-1]; idcg=(ideal*DISCOUNT).sum(1)
    return dcg/np.maximum(idcg,1e-12)
def ndcg5(yy,ss): return float(per_user_ndcg(yy,ss).mean())
def swap_values(scores,row,a,b):
    out=scores.copy(); out[row,a],out[row,b]=out[row,b],out[row,a]; return out

CHAMPION_SCORES=champion[MODULES].to_numpy(np.float32)
assert top5_hash(CHAMPION_SCORES)==CHAMPION_TOP5_SHA256
print("Champion verified; eligible S0 after V25 protection:",TEST_ELIGIBLE.sum())

DATA /kaggle/input/datasets/riantamaputra/mine-today-data-mining-competition-it-today-2026
Champion verified; eligible S0 after V25 protection: 889


In [3]:
# =========================
# 2. Deterministic text/assessment views + target-blind template geometry + validation geometry
# =========================
URL_RE=re.compile(r"https?://\S+|www\.\S+",re.I)
WS_RE=re.compile(r"\s+")
PUNCT_RE=re.compile(r"[^\w<>]+",re.UNICODE)

def norm_text(x):
    x="" if pd.isna(x) else str(x)
    x=html.unescape(unicodedata.normalize("NFKC",x)).casefold()
    x=URL_RE.sub(" <url> ",x); x=PUNCT_RE.sub(" ",x); return WS_RE.sub(" ",x).strip()

ALIASES=[
("generative ai",),("gen ai",),("genai",),("large language model",),("llm",),
("exploratory data analysis",),("eda",),("machine learning",),("ml",),
("computer vision",),("cv",),("natural language processing",),("nlp",),
("prompt engineering",),("web scraping",),("scraping",),("version control",),("git",),
("dashboard",),("mlops",),("no code ai",),("nocode",),("automation",),
("statistics",),("statistik",),("python",),("sql",),("excel",),("career",),("karir",),("portfolio",),("portofolio",)
]
ALIAS_PATTERNS=[re.compile(r"(?<!\w)"+re.escape(a[0])+r"(?!\w)") for a in sorted(ALIASES,key=lambda z:-len(z[0]))]
def skeleton(x):
    s=norm_text(x)
    for rx in ALIAS_PATTERNS: s=rx.sub(" <topic> ",s)
    s=re.sub(r"\b\d+\b"," <num> ",s); return WS_RE.sub(" ",s).strip()

chat=chat.copy(); chat["user_id"]=chat.user_id.astype(str)
chat["timestamp"]=pd.to_datetime(chat.timestamp,errors="coerce")
chat=chat.sort_values(["user_id","timestamp","chat_id"],kind="stable")
byu={u:g for u,g in chat.groupby("user_id",sort=False)}
full_text=[]; latest_text=[]; full_skel=[]; latest_skel=[]; msg_lists=[]; chat_count=[]
for u in ALL_USERS:
    g=byu.get(u)
    msgs=[] if g is None else [norm_text(t) for t in g.user_chat_text.tolist() if norm_text(t)]
    msg_lists.append(msgs); chat_count.append(len(msgs))
    latest=msgs[-1] if msgs else "<no_chat>"; full=" [sep] ".join(msgs) if msgs else "<no_chat>"
    latest_text.append(latest); full_text.append(full); latest_skel.append(skeleton(latest)); full_skel.append(skeleton(full))
full_text=np.array(full_text,dtype=object); latest_text=np.array(latest_text,dtype=object)
full_skel=np.array(full_skel,dtype=object); latest_skel=np.array(latest_skel,dtype=object); chat_count=np.asarray(chat_count,np.float32)

# Assessment parse: invalid/out-of-range values remain NaN until fold-local imputation.
WORD_NUM={"nol":0,"zero":0,"satu":1,"dua":2,"tiga":3,"empat":4,"lima":5}
def parse_obj(s):
    try: return json.loads(s)
    except Exception: return ast.literal_eval(s)
assess=assess.copy(); assess["user_id"]=assess.user_id.astype(str)
assert set(ALL_USERS).issubset(set(assess.user_id))
assess=assess.set_index("user_id").loc[ALL_USERS].reset_index()
objs=[parse_obj(s) for s in assess.assessment_result.astype(str)]
SKILLS=list(objs[0].keys()); assert len(SKILLS)==10
A_RAW=np.full((N,10),np.nan,np.float32); A_BAD=np.zeros((N,10),np.float32)
for i,o in enumerate(objs):
    for j,k in enumerate(SKILLS):
        v=o.get(k,np.nan); z=None
        if isinstance(v,str) and v.strip().casefold() in WORD_NUM: z=float(WORD_NUM[v.strip().casefold()])
        else:
            try: z=float(v)
            except Exception: z=np.nan
        if np.isfinite(z) and abs(z-round(z))<1e-8 and 0<=z<=5: A_RAW[i,j]=z
        else: A_BAD[i,j]=1.0
assert int(A_BAD.sum())==251, f"assessment parser drift: {A_BAD.sum()} invalid cells"

# Module documents and readiness map from organizer catalog semantics.
mc=modules.set_index("module_id").loc[MODULES]
MOD_TEXT=(mc.module_name.astype(str)+" [sep] "+mc.description_and_syllabus.astype(str)+" [sep] "+mc.prerequisite_level.astype(str)).map(norm_text).tolist()
MOD_SKILLS={
0:[],1:[0],2:[1],3:[0],4:[9],5:[2],6:[0,2,3],7:[1],8:[2,3,4,5],9:[4,6],10:[4],11:[4,7],12:[7],13:[7,8],14:[7],15:[0,4,5,9],16:[8,9]
}

# Target-blind near-template components. No-chat users are deliberately unique groups.
vec_grp=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=1,max_features=50000,norm="l2")
Xg=vec_grp.fit_transform(full_skel.tolist())
sim=(Xg@Xg.T).tocsr(); sim.data=(sim.data>=cfg.near_threshold).astype(np.int8); sim.eliminate_zeros()
_,near_group=connected_components(sim,directed=False)
nochat=np.array([t=="<no_chat>" for t in full_text])
base_gid=int(near_group.max())+1
near_group=near_group.astype(np.int64); near_group[nochat]=base_gid+np.flatnonzero(nochat)
ALL_TRAIN=np.arange(NTR,dtype=int)
ALL_TEST=np.arange(NTR,N,dtype=int)

# Manifest is target-free. It is diagnostic only and never used as a test label proxy.
pd.DataFrame({
    "user_id":train.user_id.astype(str),
    "near_group":near_group[:NTR],
    "latest_skeleton":latest_skel[:NTR],
    "full_history_sha256":[hashlib.sha256(str(t).encode()).hexdigest() for t in full_text[:NTR]],
}).to_csv(OUT/"v129_template_manifest.csv",index=False)
print("template geometry ready; near groups(train/all)=",len(np.unique(near_group[:NTR])),len(np.unique(near_group)))
del Xg,sim,vec_grp; gc.collect()


template geometry ready; near groups(train/all)= 1912 2297


0

In [4]:
# =========================
# 3. Frozen dense representations (E5 + Nomic), cache-provenance guarded
# =========================
E5_ID="intfloat/multilingual-e5-base"
NOMIC_ID="nomic-ai/nomic-embed-text-v2-moe"

def text_sha(texts,prefix):
    h=hashlib.sha256(); h.update(prefix.encode())
    for t in texts: h.update(b"\0"); h.update(str(t).encode("utf-8"))
    return h.hexdigest()

def _resolve_model(model_id):
    # Prefer an attached Kaggle model snapshot when internet is disabled.
    needles=[model_id.split("/")[-1].lower().replace("_","-")]
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob("config.json"):
            s=str(p.parent).lower().replace("_","-")
            if any(n in s for n in needles): return str(p.parent)
    return model_id

def load_or_encode_e5(texts,prefix,name,device="cuda:0"):
    key=text_sha(texts,E5_ID+"|"+prefix); f=OUT/f"{name}_e5.npy"; meta=OUT/f"{name}_e5.json"
    if f.exists() and meta.exists():
        m=json.loads(meta.read_text()); arr=np.load(f)
        if m.get("key")==key and arr.shape[0]==len(texts): return np.asarray(arr,np.float32)
    import torch
    from sentence_transformers import SentenceTransformer
    mid=_resolve_model(E5_ID); model=SentenceTransformer(mid,device=device)
    arr=np.asarray(model.encode([prefix+t for t in texts],batch_size=64,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True),np.float32)
    np.save(f,arr); meta.write_text(json.dumps({"key":key,"model":E5_ID,"resolved":mid,"prefix":prefix,"shape":list(arr.shape)},indent=2))
    del model; gc.collect(); torch.cuda.empty_cache(); return arr

def load_or_encode_nomic(texts,prefix,name,device="cuda:1"):
    key=text_sha(texts,NOMIC_ID+"|"+prefix); f=OUT/f"{name}_nomic.npy"; meta=OUT/f"{name}_nomic.json"
    if f.exists() and meta.exists():
        m=json.loads(meta.read_text()); arr=np.load(f)
        if m.get("key")==key and arr.shape[0]==len(texts): return np.asarray(arr,np.float32)
    import torch
    from transformers import AutoTokenizer,AutoModel
    mid=_resolve_model(NOMIC_ID); tok=AutoTokenizer.from_pretrained(mid,trust_remote_code=True)
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
    model=AutoModel.from_pretrained(mid,trust_remote_code=True,torch_dtype=dtype).to(device).eval()
    rows=[]
    with torch.no_grad():
        for a in range(0,len(texts),48):
            batch=[prefix+t for t in texts[a:a+48]]
            e=tok(batch,padding=True,truncation=True,max_length=256,return_tensors="pt"); e={k:v.to(device) for k,v in e.items()}
            h=model(**e).last_hidden_state; mask=e["attention_mask"].unsqueeze(-1).float()
            v=((h*mask).sum(1)/mask.sum(1).clamp_min(1)).float(); v=v/v.norm(dim=1,keepdim=True).clamp_min(1e-12)
            rows.append(v.cpu().numpy())
    arr=np.concatenate(rows).astype(np.float32); np.save(f,arr)
    meta.write_text(json.dumps({"key":key,"model":NOMIC_ID,"resolved":mid,"prefix":prefix,"shape":list(arr.shape)},indent=2))
    del model; gc.collect(); torch.cuda.empty_cache(); return arr

if not __import__('torch').cuda.is_available():
    print("WARNING: CUDA unavailable; encoding will be slow. Production target is T4x2.")
    dev_e5=dev_no="cpu"
else:
    ng=__import__('torch').cuda.device_count(); dev_e5="cuda:0"; dev_no="cuda:1" if ng>1 else "cuda:0"

E5U=load_or_encode_e5(full_text.tolist(),"query: ","users",dev_e5)
E5M=load_or_encode_e5(MOD_TEXT,"passage: ","modules",dev_e5)
NOU=load_or_encode_nomic(full_text.tolist(),"search_query: ","users",dev_no)
NOM=load_or_encode_nomic(MOD_TEXT,"search_document: ","modules",dev_no)
assert E5U.shape[0]==N and NOU.shape[0]==N and E5M.shape[0]==NOM.shape[0]==M
E5_MOD=(E5U@E5M.T).astype(np.float32); NOM_MOD=(NOU@NOM.T).astype(np.float32)
print("dense ready",E5U.shape,NOU.shape)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- configuration_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


modeling_hf_nomic_bert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- modeling_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.90G [00:00<?, ?B/s]

dense ready (5000, 768) (5000, 768)


In [5]:
# =========================
# 4. Lean label-free candidate features: sparse + dense + assessments + anchor geometry
# =========================
def fit_bundle(ref_idx):
    ref_idx=np.asarray(ref_idx,int)
    corpus=[full_text[i] for i in ref_idx]+MOD_TEXT
    wv=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=cfg.word_max_features,sublinear_tf=True,smooth_idf=True,norm="l2")
    cv=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=2,max_features=cfg.char_max_features,sublinear_tf=True,norm="l2")
    wv.fit(corpus); cv.fit(corpus)
    return {
        "wv":wv,"cv":cv,
        "wfull":wv.transform(full_text.tolist()),"wlatest":wv.transform(latest_text.tolist()),"wm":wv.transform(MOD_TEXT),
        "cfull":cv.transform(full_text.tolist()),"clatest":cv.transform(latest_text.tolist()),"cm":cv.transform(MOD_TEXT)
    }

def recency_similarity(bundle,indices,key="word"):
    indices=np.asarray(indices,int); out=np.zeros((len(indices),M),np.float32)
    vec=bundle["wv"] if key=="word" else bundle["cv"]
    mm=bundle["wm"] if key=="word" else bundle["cm"]
    for qi,g in enumerate(indices):
        msgs=msg_lists[g]
        if not msgs: continue
        S=(vec.transform(msgs)@mm.T).toarray().astype(np.float32)
        w=np.power(0.5,np.arange(len(msgs)-1,-1,-1,dtype=np.float32)); w/=w.sum()
        out[qi]=(w[:,None]*S).sum(0)
    return out

def geom_from_scores(S,prefix):
    S=np.asarray(S,np.float32); order=np.argsort(-S,axis=1,kind="stable"); ranks=np.empty_like(order)
    rr=np.arange(len(S))[:,None]; ranks[rr,order]=np.arange(1,M+1)[None,:]
    best=S.max(1,keepdims=True); fifth=np.take_along_axis(S,order[:,4:5],1)
    return [(prefix+"_score",S),(prefix+"_rank",ranks.astype(np.float32)/M),(prefix+"_gapbest",best-S),(prefix+"_gap5",S-fifth)]

def impute_assessment(ref_idx,q_idx):
    ref_idx=np.asarray(ref_idx,int); q_idx=np.asarray(q_idx,int)
    med=np.nanmedian(A_RAW[ref_idx],axis=0); med=np.where(np.isfinite(med),med,2.5)
    A=np.where(np.isfinite(A_RAW[q_idx]),A_RAW[q_idx],med[None,:]).astype(np.float32)
    return A,A_BAD[q_idx].astype(np.float32)

def readiness(A,BAD):
    R=np.zeros((len(A),M),np.float32); I=np.zeros_like(R)
    for m,js in MOD_SKILLS.items():
        if js: R[:,m]=A[:,js].mean(1); I[:,m]=BAD[:,js].mean(1)
        else: R[:,m]=A.mean(1); I[:,m]=BAD.mean(1)
    return R,I

def candidate_features(bundle,ref_idx,q_idx,anchor_q):
    ref_idx=np.asarray(ref_idx,int); q_idx=np.asarray(q_idx,int); pieces=[]
    wf=(bundle["wfull"][q_idx]@bundle["wm"].T).toarray().astype(np.float32)
    wl=(bundle["wlatest"][q_idx]@bundle["wm"].T).toarray().astype(np.float32)
    wr=recency_similarity(bundle,q_idx,"word")
    cf=(bundle["cfull"][q_idx]@bundle["cm"].T).toarray().astype(np.float32)
    cl=(bundle["clatest"][q_idx]@bundle["cm"].T).toarray().astype(np.float32)
    cr=recency_similarity(bundle,q_idx,"char")
    for p,S in [
        ("tfidf_word_full",wf),("tfidf_word_latest",wl),("tfidf_word_recency",wr),
        ("tfidf_char_full",cf),("tfidf_char_latest",cl),("tfidf_char_recency",cr),
        ("e5_mod",E5_MOD[q_idx]),("nomic_mod",NOM_MOD[q_idx]),
    ]: pieces.extend(geom_from_scores(S,p))
    pieces += [("dense_e5_minus_nomic",E5_MOD[q_idx]-NOM_MOD[q_idx])]

    Aq,Bq=impute_assessment(ref_idx,q_idx); R,I=readiness(Aq,Bq)
    pieces += [("ready",R),("ready_invalid",I)]
    for j,s in enumerate(SKILLS):
        pieces.append((f"assess_{j:02d}",np.repeat(Aq[:,j:j+1],M,1)))
        pieces.append((f"assess_bad_{j:02d}",np.repeat(Bq[:,j:j+1],M,1)))
    pieces += [
        ("assess_mean",np.repeat(Aq.mean(1,keepdims=True),M,1)),
        ("assess_std",np.repeat(Aq.std(1,keepdims=True),M,1)),
        ("assess_invalid_count",np.repeat(Bq.sum(1,keepdims=True),M,1)),
        ("chat_count",np.repeat(chat_count[q_idx,None],M,1)),
        ("no_chat",np.repeat((chat_count[q_idx]==0).astype(np.float32)[:,None],M,1)),
    ]
    pieces.extend(geom_from_scores(anchor_q,"anchor"))

    names=[]; arrays=[]
    for n,a in pieces:
        a=np.asarray(a,np.float32); assert a.shape==(len(q_idx),M),(n,a.shape)
        names.append(n); arrays.append(a[:,:,None])
    X=np.concatenate(arrays,2).astype(np.float32)
    assert np.isfinite(X).all()
    # V129 contract: no BM25 and no target-derived KNN/template statistics.
    assert not any(("bm25" in n) or n.startswith("knn_") or n.startswith("tpl_") for n in names)
    return X,names


In [6]:
# =========================
# 5. Sparse anchor analogue + per-module LightGBM Stage-1
# =========================
def make_inner_splits(indices,seed):
    indices=np.asarray(indices,int); groups=near_group[indices]; strat=np.argmax(y[indices],1)
    try:
        sg=StratifiedGroupKFold(cfg.n_inner,shuffle=True,random_state=seed)
        return [(indices[a],indices[b]) for a,b in sg.split(np.zeros(len(indices)),strat,groups)]
    except Exception:
        gg=GroupKFold(cfg.n_inner)
        return [(indices[a],indices[b]) for a,b in gg.split(np.zeros(len(indices)),groups=groups)]

def _anchor_matrix(bundle,ref_idx,q_idx,med_ref=None):
    ref_idx=np.asarray(ref_idx,int); q_idx=np.asarray(q_idx,int)
    if med_ref is None:
        med_ref=np.nanmedian(A_RAW[ref_idx],axis=0); med_ref=np.where(np.isfinite(med_ref),med_ref,2.5)
    Aq=np.where(np.isfinite(A_RAW[q_idx]),A_RAW[q_idx],med_ref[None,:]).astype(np.float32)
    B=A_BAD[q_idx].astype(np.float32)
    extra=np.c_[Aq,B,chat_count[q_idx],(chat_count[q_idx]==0).astype(np.float32)]
    return hstack([bundle["wfull"][q_idx],bundle["cfull"][q_idx],csr_matrix(extra)],format="csr")

def nested_anchor(bundle,train_idx,query_idx,inner_splits):
    train_idx=np.asarray(train_idx,int); query_idx=np.asarray(query_idx,int); pos={g:i for i,g in enumerate(train_idx)}
    oof_rel=np.zeros((len(train_idx),M),np.float32); oof_top=np.zeros_like(oof_rel)
    for f,(tr,va) in enumerate(inner_splits):
        med=np.nanmedian(A_RAW[tr],axis=0); med=np.where(np.isfinite(med),med,2.5)
        Xtr=_anchor_matrix(bundle,tr,tr,med); Xv=_anchor_matrix(bundle,tr,va,med)
        r=Ridge(alpha=3.0,solver="lsqr",tol=2e-3,max_iter=1000).fit(Xtr,y[tr])
        t=Ridge(alpha=3.0,solver="lsqr",tol=2e-3,max_iter=1000).fit(Xtr,np.eye(M,dtype=np.float32)[np.argmax(y[tr],1)])
        loc=np.array([pos[g] for g in va]); oof_rel[loc]=r.predict(Xv); oof_top[loc]=t.predict(Xv)
    med=np.nanmedian(A_RAW[train_idx],axis=0); med=np.where(np.isfinite(med),med,2.5)
    Xtr=_anchor_matrix(bundle,train_idx,train_idx,med); Xq=_anchor_matrix(bundle,train_idx,query_idx,med)
    r=Ridge(alpha=3.0,solver="lsqr",tol=2e-3,max_iter=1000).fit(Xtr,y[train_idx])
    t=Ridge(alpha=3.0,solver="lsqr",tol=2e-3,max_iter=1000).fit(Xtr,np.eye(M,dtype=np.float32)[np.argmax(y[train_idx],1)])
    oof=(.9*oof_rel+.1*oof_top).astype(np.float32); q=(.9*r.predict(Xq)+.1*t.predict(Xq)).astype(np.float32)
    return oof,q

def fit_lgb_stage1(Xtr,ytr,Xq):
    assert Xtr.ndim==3 and Xq.ndim==3 and Xtr.shape[1]==Xq.shape[1]==M and Xtr.shape[2]==Xq.shape[2]
    pred=np.zeros((len(Xq),M),np.float32)
    for m in range(M):
        vals=[]
        for seed in cfg.lgb_seeds:
            model=lgb.LGBMRegressor(
                objective="regression_l2",n_estimators=cfg.lgb_estimators,learning_rate=cfg.lgb_lr,
                num_leaves=cfg.lgb_num_leaves,max_depth=cfg.lgb_max_depth,min_child_samples=cfg.lgb_min_child,
                subsample=cfg.lgb_subsample,subsample_freq=1,colsample_bytree=cfg.lgb_colsample,
                reg_alpha=cfg.lgb_reg_alpha,reg_lambda=cfg.lgb_reg_lambda,
                random_state=int(seed+37*m),n_jobs=2,verbosity=-1,force_col_wise=True)
            model.fit(Xtr[:,m,:],ytr[:,m]); vals.append(model.predict(Xq[:,m,:]))
        pred[:,m]=np.mean(vals,axis=0)
    return pred

def fit_stage1_once(train_idx,query_idx,bundle,seed):
    """Train Stage-1 without using any query labels. `bundle` may include query text vocabulary only label-free."""
    train_idx=np.asarray(train_idx,int); query_idx=np.asarray(query_idx,int)
    inner=make_inner_splits(train_idx,seed+11)
    anchor_tr,anchor_q=nested_anchor(bundle,train_idx,query_idx,inner)
    Xtr,names=candidate_features(bundle,train_idx,train_idx,anchor_tr)
    Xq,nq=candidate_features(bundle,train_idx,query_idx,anchor_q); assert nq==names
    pred=fit_lgb_stage1(Xtr,y[train_idx],Xq)
    assert np.isfinite(pred).all()
    del Xtr; gc.collect()
    return {"anchor_query":anchor_q,"fusion_query":pred,"raw_query":Xq,"names":names}

def make_world_splits(indices,world,seed=SEED):
    indices=np.asarray(indices,int)
    strat=np.array([f"{np.argmax(y[i])}_{int((y[i]>0).sum())}" for i in indices],object)
    cc=Counter(strat); strat=np.array([s if cc[s]>=cfg.n_outer else str(np.argmax(y[i])) for s,i in zip(strat,indices)],object)
    if world=="random":
        sk=StratifiedKFold(cfg.n_outer,shuffle=True,random_state=seed)
        return [(indices[a],indices[b]) for a,b in sk.split(np.zeros(len(indices)),strat)]
    if world in {"exact","latest","near"}:
        groups={"exact":full_text[indices],"latest":latest_skel[indices],"near":near_group[indices]}[world]
        try:
            sg=StratifiedGroupKFold(cfg.n_outer,shuffle=True,random_state=seed)
            return [(indices[a],indices[b]) for a,b in sg.split(np.zeros(len(indices)),strat,groups)]
        except Exception:
            gg=GroupKFold(cfg.n_outer)
            return [(indices[a],indices[b]) for a,b in gg.split(np.zeros(len(indices)),groups=groups)]
    if world=="contiguous":
        order=indices[np.argsort([int(re.sub(r"\D","",ALL_USERS[i])) for i in indices])]
        blocks=np.array_split(order,cfg.n_outer)
        return [(np.setdiff1d(indices,b),np.asarray(b,int)) for b in blocks]
    raise ValueError(world)


In [7]:
# =========================
# 6. Exact ΔNDCG action representation + fixed-coverage policy
# =========================
def action_table(indices,anchor,fusion,raw,names):
    indices=np.asarray(indices,int); n=len(indices); X=[]; Y=[]; users=[]; cand=[]; inc=[]; ranks=[]
    order=np.argsort(-anchor,axis=1,kind="stable")
    for i,g in enumerate(indices):
        im=int(order[i,0])
        for rr in (1,2,3):
            cm=int(order[i,rr]); s=anchor[i].copy()
            if g < NTR:
                before=per_user_ndcg(y[g:g+1],s[None,:])[0]
                s[im],s[cm]=s[cm],s[im]
                delta=float(per_user_ndcg(y[g:g+1],s[None,:])[0]-before)
            else: delta=0.0
            diff=raw[i,cm]-raw[i,im]
            pair=np.zeros(M*M,np.float32); pair[im*M+cm]=1.0
            c1=np.zeros(M,np.float32); c1[cm]=1.0
            i1=np.zeros(M,np.float32); i1[im]=1.0
            r1=np.zeros(3,np.float32); r1[rr-1]=1.0
            feat=np.r_[fusion[i,cm],fusion[i,im],fusion[i,cm]-fusion[i,im],anchor[i,im]-anchor[i,cm],diff,c1,i1,pair,r1]
            X.append(feat); Y.append(delta); users.append(i); cand.append(cm); inc.append(im); ranks.append(rr+1)
    fn=["fusion_c","fusion_i","fusion_diff","anchor_margin"]+["d_"+x for x in names]+[f"cand_{m}" for m in MODULES]+[f"inc_{m}" for m in MODULES]+[f"pair_{a}_{b}" for a in MODULES for b in MODULES]+["rank2","rank3","rank4"]
    X=np.asarray(X,np.float32); Y=np.asarray(Y,np.float32)
    assert X.shape[0]==3*n and np.isfinite(X).all() and np.isfinite(Y).all()
    return X,Y,np.asarray(users,int),np.asarray(cand,int),np.asarray(inc,int),np.asarray(ranks,int),fn

def fit_action_model(X,Y):
    model=make_pipeline(StandardScaler(),Ridge(alpha=cfg.action_alpha,solver="lsqr",tol=1e-5,max_iter=3000))
    model.fit(X,Y); return model

def predict_action(model,indices,anchor,fusion,raw,names):
    X,Y,U,C,I,R,fn=action_table(indices,anchor,fusion,raw,names)
    return {"X":X,"y":Y,"user":U,"cand":C,"inc":I,"rank":R,"pred":model.predict(X).astype(np.float32),"features":fn}

def bootstrap_ci(d,groups=None,seed=SEED):
    d=np.asarray(d,float); rng=np.random.default_rng(seed); vals=np.empty(cfg.bootstraps,float)
    if groups is None:
        n=len(d)
        for b in range(cfg.bootstraps): vals[b]=d[rng.integers(0,n,n)].mean()
    else:
        groups=np.asarray(groups); ug=np.unique(groups); bucket={g:np.flatnonzero(groups==g) for g in ug}
        for b in range(cfg.bootstraps):
            pick=rng.choice(ug,len(ug),replace=True); ix=np.concatenate([bucket[g] for g in pick]); vals[b]=d[ix].mean()
    return float(np.quantile(vals,.025)),float(np.quantile(vals,.975))

def fixed_policy_from_arrays(pred,y_delta,cand,n_users,coverage,eligible=None):
    P=np.asarray(pred).reshape(n_users,3); D=np.asarray(y_delta).reshape(n_users,3); C=np.asarray(cand).reshape(n_users,3)
    j=np.argmax(P,1); score=P[np.arange(n_users),j]; realized=D[np.arange(n_users),j]; bc=C[np.arange(n_users),j]
    ok=score>0
    if eligible is not None: ok &= np.asarray(eligible,bool)
    ids=np.flatnonzero(ok); k=max(1,int(math.ceil(float(coverage)*n_users)))
    pick=ids[np.argsort(-score[ids],kind="stable")[:min(k,len(ids))]]
    sel=np.zeros(n_users,bool); sel[pick]=True; ud=np.zeros(n_users,float); ud[pick]=realized[pick]
    return sel,bc,score,ud

def fixed_policy(act,n_users,coverage=None,eligible=None,groups=None,fold_id=None):
    coverage=cfg.fixed_coverage if coverage is None else float(coverage)
    sel,bc,score,ud=fixed_policy_from_arrays(act["pred"],act["y"],act["cand"],n_users,coverage,eligible)
    ciu=bootstrap_ci(ud,seed=SEED+11); cig=bootstrap_ci(ud,groups,SEED+12) if groups is not None else ciu
    out={"requested":coverage,"coverage":float(sel.mean()),"gain":float(ud.mean()),"n":int(sel.sum()),
         "benefit":int((ud>0).sum()),"harm":int((ud<0).sum()),"tie":int(sel.sum()-(ud!=0).sum()),
         "ci_user":ciu,"ci_cluster":cig,"selected":sel,"candidate":bc,"score":score,"user_delta":ud}
    if fold_id is not None:
        out["fold_gains"]=[float(ud[fold_id==f].mean()) for f in sorted(np.unique(fold_id))]
    return out

# Cheap invariant tests before expensive training.
assert abs(ndcg5(y[:5],y[:5])-1.0)<1e-8
_tmp_pred=np.array([[3,2,1],[1,3,2]],np.float32).reshape(-1)
_tmp_y=np.array([[.1,.2,.3],[.4,.5,.6]],np.float32).reshape(-1)
_tmp_c=np.tile(np.arange(3),2)
_tmp_sel,_,_,_=fixed_policy_from_arrays(_tmp_pred,_tmp_y,_tmp_c,2,.5)
assert _tmp_sel.sum()==1


In [8]:
# =========================
# 7. TRUE second-level nesting: one evaluation world
# =========================
def fully_nested_world(indices,world):
    indices=np.asarray(indices,int); local={g:i for i,g in enumerate(indices)}
    wseed=cfg.seed+int(hashlib.sha1(world.encode()).hexdigest()[:6],16)%10000
    outer=make_world_splits(indices,world,wseed)
    n=len(indices); fold_id=np.full(n,-1,int); seen=np.zeros(n,bool)
    action_pred=np.zeros((n,3),np.float32); action_y=np.zeros((n,3),np.float32); action_c=np.zeros((n,3),np.int16); action_i=np.zeros((n,3),np.int16)
    oof_action_X=None; action_feature_names=None; stage1_fusion=np.zeros((n,M),np.float32); stage1_anchor=np.zeros((n,M),np.float32)
    fold_rows=[]

    for f,(tr,va) in enumerate(outer):
        t0=time.time(); bundle=fit_bundle(tr)
        # Inner action-training rows: each user's Stage-1 prediction is produced by a model that never sees that user's labels.
        inner=make_inner_splits(tr,wseed+100+f); pos={g:i for i,g in enumerate(tr)}
        atr=np.zeros((len(tr),M),np.float32); ftr=np.zeros_like(atr); rtr=None; names=None
        for j,(ir,iv) in enumerate(inner):
            z=fit_stage1_once(ir,iv,bundle,wseed+10000*f+100*j)
            loc=np.array([pos[g] for g in iv]); atr[loc]=z["anchor_query"]; ftr[loc]=z["fusion_query"]
            if rtr is None: rtr=np.zeros((len(tr),M,z["raw_query"].shape[2]),np.float32); names=z["names"]
            assert z["names"]==names; rtr[loc]=z["raw_query"]
            del z; gc.collect()
        assert np.isfinite(atr).all() and np.isfinite(ftr).all() and np.isfinite(rtr).all()
        Xa,Ya,Ua,Ca,Ia,Ra,fn=action_table(tr,atr,ftr,rtr,names)
        amodel=fit_action_model(Xa,Ya)

        # Outer validation: labels are untouched until after all Stage-1 and action-head predictions exist.
        zv=fit_stage1_once(tr,va,bundle,wseed+50000+f)
        av=predict_action(amodel,va,zv["anchor_query"],zv["fusion_query"],zv["raw_query"],zv["names"])
        lv=np.array([local[g] for g in va]); fold_id[lv]=f
        action_pred[lv]=av["pred"].reshape(len(va),3); action_y[lv]=av["y"].reshape(len(va),3)
        action_c[lv]=av["cand"].reshape(len(va),3); action_i[lv]=av["inc"].reshape(len(va),3)
        stage1_fusion[lv]=zv["fusion_query"]; stage1_anchor[lv]=zv["anchor_query"]
        seen[lv]=np.isin(latest_skel[va],np.unique(latest_skel[tr]))

        # Store the clean OUTER-validation action feature rows for the eventual all-data action model.
        if oof_action_X is None:
            oof_action_X=np.zeros((3*n,av["X"].shape[1]),np.float32); action_feature_names=av["features"]
        assert av["features"]==action_feature_names
        for q,li in enumerate(lv): oof_action_X[3*li:3*li+3]=av["X"][3*q:3*q+3]

        fold_rows.append({"world":world,"fold":f,"n":len(va),"anchor_ndcg":ndcg5(y[va],zv["anchor_query"]),
                          "fusion_ndcg":ndcg5(y[va],zv["fusion_query"]),"seconds":time.time()-t0})
        del Xa,Ya,Ua,Ca,Ia,Ra,amodel,zv,av,rtr,bundle; gc.collect()

    assert (fold_id>=0).all() and oof_action_X is not None and np.isfinite(oof_action_X).all()
    flat_pred=action_pred.reshape(-1); flat_y=action_y.reshape(-1); flat_c=action_c.reshape(-1)
    act={"pred":flat_pred,"y":flat_y,"cand":flat_c}
    pol=fixed_policy(act,n,cfg.fixed_coverage,groups=near_group[indices],fold_id=fold_id)
    ud=pol["user_delta"]
    sg=float(ud[seen].mean()) if seen.any() else 0.0; ug=float(ud[~seen].mean()) if (~seen).any() else 0.0
    mix=cfg.seen_weight*sg+(1-cfg.seen_weight)*ug

    # Coverage curve is diagnostic only; fixed 5% remains primary no matter which row looks best.
    curve=[]
    for c in cfg.diagnostic_coverages:
        z=fixed_policy(act,n,c,groups=near_group[indices],fold_id=fold_id)
        curve.append({"world":world,"requested":c,"gain":z["gain"],"coverage":z["coverage"],"n":z["n"],
                      "benefit":z["benefit"],"harm":z["harm"],"ci_user_low":z["ci_user"][0],"ci_user_high":z["ci_user"][1],
                      "ci_cluster_low":z["ci_cluster"][0],"ci_cluster_high":z["ci_cluster"][1]})

    return {"world":world,"indices":indices,"fold_id":fold_id,"seen":seen,"policy":pol,"seen_gain":sg,"unseen_gain":ug,"mix_gain":float(mix),
            "fold_metrics":pd.DataFrame(fold_rows),"curve":pd.DataFrame(curve),"action_pred":action_pred,"action_y":action_y,
            "action_cand":action_c,"action_inc":action_i,"oof_action_X":oof_action_X,"oof_action_y":flat_y,
            "action_feature_names":action_feature_names,"stage1_fusion":stage1_fusion,"stage1_anchor":stage1_anchor}


In [9]:
# =========================
# 8. PRIMARY random-user validation, fixed 5% policy, benchmark-faithful seen/unseen mixture
# =========================
t0=time.time(); primary=fully_nested_world(ALL_TRAIN,"random")
pp=primary["policy"]; pos_folds=int((np.asarray(pp["fold_gains"])>0).sum())
print("PRIMARY fixed-5% gain",pp["gain"],"mix",primary["mix_gain"],"seen/unseen",primary["seen_gain"],primary["unseen_gain"],"positive folds",pos_folds)
print("CI user",pp["ci_user"],"CI near-cluster",pp["ci_cluster"],"actions",pp["n"],"benefit/harm",pp["benefit"],pp["harm"])
primary["fold_metrics"].to_csv(OUT/"v129_primary_fold_metrics.csv",index=False)
primary["curve"].to_csv(OUT/"v129_primary_coverage_diagnostic.csv",index=False)
json.dump(primary["action_feature_names"],open(OUT/"v129_action_feature_names.json","w"),indent=2)

# Per-user primary ledger. Benefit:harm is reported, never used as a gate.
P=primary["action_pred"]; C=primary["action_cand"]; I=primary["action_inc"]; j=np.argmax(P,1)
ledger=pd.DataFrame({
    "user_id":train.user_id.astype(str),"fold":primary["fold_id"],"seen_latest_skeleton":primary["seen"],"near_group":near_group[:NTR],
    "selected":pp["selected"],"candidate":np.array(MODULES)[C[np.arange(NTR),j]],"incumbent":np.array(MODULES)[I[np.arange(NTR),j]],
    "pred_utility":P[np.arange(NTR),j],"realized_delta":pp["user_delta"]})
ledger.to_csv(OUT/"v129_primary_action_ledger.csv",index=False)
np.savez_compressed(OUT/"v129_primary_oof.npz",stage1_fusion=primary["stage1_fusion"],stage1_anchor=primary["stage1_anchor"],
                    action_pred=primary["action_pred"],action_y=primary["action_y"],fold_id=primary["fold_id"],seen=primary["seen"])
print("primary completed in",round(time.time()-t0,1),"s")


PRIMARY fixed-5% gain 0.0011301930556073786 mix 0.0011703919159089214 seen/unseen 0.0011925042934738958 0.0010723283284468615 positive folds 5
CI user (0.00036428423765464687, 0.0018839645147905682) CI near-cluster (0.0003157520465789268, 0.0019904358119389163) actions 200 benefit/harm 112 78
primary completed in 433.8 s


In [10]:
# =========================
# 9. Secondary validation worlds — diagnostics, not one-by-one hard vetoes
# =========================
world_rows=[{"world":"random","gain":primary["policy"]["gain"],"mix_gain":primary["mix_gain"],"seen_gain":primary["seen_gain"],"unseen_gain":primary["unseen_gain"],
             "coverage":primary["policy"]["coverage"],"n":primary["policy"]["n"],"positive_folds":int((np.asarray(primary["policy"]["fold_gains"])>0).sum()),
             "worst_fold":float(np.min(primary["policy"]["fold_gains"]))}]
curve_frames=[primary["curve"]]
if cfg.run_stress:
    for world in ["exact","latest","near","contiguous"]:
        t=time.time(); z=fully_nested_world(ALL_TRAIN,world); p=z["policy"]
        world_rows.append({"world":world,"gain":p["gain"],"mix_gain":z["mix_gain"],"seen_gain":z["seen_gain"],"unseen_gain":z["unseen_gain"],
                           "coverage":p["coverage"],"n":p["n"],"positive_folds":int((np.asarray(p["fold_gains"])>0).sum()),"worst_fold":float(np.min(p["fold_gains"]))})
        curve_frames.append(z["curve"]); z["fold_metrics"].to_csv(OUT/f"v129_{world}_fold_metrics.csv",index=False)
        print(world,"gain",p["gain"],"mix",z["mix_gain"],"seconds",round(time.time()-t,1))
        del z; gc.collect()
worlds=pd.DataFrame(world_rows); worlds.to_csv(OUT/"v129_world_metrics.csv",index=False)
pd.concat(curve_frames,ignore_index=True).to_csv(OUT/"v129_all_world_coverage_diagnostic.csv",index=False)
display(worlds)

# Competition-oriented decision layer. Only catastrophic systematic stress influences status; ordinary negative diagnostics are reported, not vetoed.
primary_mix=float(primary["mix_gain"]); primary_pos=int((np.asarray(primary["policy"]["fold_gains"])>0).sum())
severe_stress=bool(cfg.run_stress and (worlds.loc[worlds.world!="random","gain"]<cfg.catastrophic_stress).any())
if (primary_mix>=cfg.green_gain) and (primary_pos>=4) and not severe_stress:
    STATUS="GREEN"
elif (primary_mix>=cfg.yellow_gain) and (primary_pos>=3) and not severe_stress:
    STATUS="YELLOW"
elif (primary_mix>0) and (primary_pos>=3):
    STATUS="BORDERLINE"
else:
    STATUS="RED"
RECOMMEND_SUBMIT=bool(STATUS in {"GREEN","YELLOW"} and not cfg.smoke)
print("V129 evidence status:",STATUS,"recommend_submit:",RECOMMEND_SUBMIT,"severe_stress:",severe_stress)


exact gain 0.0006454438357613981 mix 0.0008239895306213228 seconds 442.7
latest gain 0.0008202871782705187 mix 0.00015093284080177548 seconds 440.9
near gain 0.0011577636846341192 mix 0.0015547202023507203 seconds 439.2
contiguous gain 0.001023684595245868 mix 0.0008105446087280541 seconds 431.3


,world,gain,mix_gain,seen_gain,unseen_gain,coverage,n,positive_folds,worst_fold
0,random,0.001130,0.001170,0.001193,0.001072,0.05,200,5,0.000026
1,exact,0.000645,0.000824,0.000887,0.000543,0.05,200,3,-0.000454
2,latest,0.000820,0.000151,0.000000,0.000820,0.05,200,5,0.000038
3,near,0.001158,0.001555,0.001686,0.000974,0.05,200,5,0.000307
4,contiguous,0.001024,0.000811,0.000691,0.001339,0.05,200,4,-0.000163


V129 evidence status: GREEN recommend_submit: True severe_stress: False


In [11]:
# =========================
# 10. Final all-train Stage-1 + clean OOF-trained action head -> exact champion fixed-5% S0 Top1 candidate
# =========================
t0=time.time(); test_idx=np.arange(NTR,N,dtype=int)
bundle_all=fit_bundle(ALL_TRAIN)
ztest=fit_stage1_once(ALL_TRAIN,test_idx,bundle_all,cfg.seed+90000)

# Final action model trains ONLY on fully out-of-fold primary action features/labels.
final_action_model=fit_action_model(primary["oof_action_X"],primary["oof_action_y"])
test_act=predict_action(final_action_model,test_idx,CHAMPION_SCORES,ztest["fusion_query"],ztest["raw_query"],ztest["names"])
assert test_act["features"]==primary["action_feature_names"],"action feature contract drift"
sel,cand,score,_=fixed_policy_from_arrays(test_act["pred"],test_act["y"],test_act["cand"],NTE,cfg.fixed_coverage,TEST_ELIGIBLE)
I=test_act["inc"].reshape(NTE,3); P=test_act["pred"].reshape(NTE,3); J=np.argmax(P,1); inc=I[np.arange(NTE),J]

cand_scores=CHAMPION_SCORES.copy(); champ_order=np.argsort(-CHAMPION_SCORES,axis=1,kind="stable")
for r in np.flatnonzero(sel):
    a=int(inc[r]); b=int(cand[r]); assert a==int(champ_order[r,0]) and b in set(map(int,champ_order[r,1:4]))
    cand_scores[r,a],cand_scores[r,b]=cand_scores[r,b],cand_scores[r,a]

candidate_path=OUT/"candidate_v129_DIAGNOSTIC.csv"; primary_path=OUT/"submission_v129_primary.csv"
out=champion.copy(); out.loc[:,MODULES]=cand_scores; out.to_csv(candidate_path,index=False)

# Hard correctness gate only: protection, score multiset, Top5 membership/rank5, schema, finite values, fixed action cap.
ta,tb=top5_idx(CHAMPION_SCORES),top5_idx(cand_scores); changed=np.any(ta!=tb,axis=1)
membership=np.array([set(a)!=set(b) for a,b in zip(ta,tb)])
locked=~TEST_ELIGIBLE; exact_locked=bool(np.array_equal(cand_scores[locked],CHAMPION_SCORES[locked]))
rank5_same=bool(np.array_equal(ta[:,4],tb[:,4])); multiset_ok=True
for r in np.flatnonzero(changed):
    if not np.array_equal(np.sort(CHAMPION_SCORES[r]),np.sort(cand_scores[r])): multiset_ok=False; break
finite_ok=bool(np.isfinite(cand_scores).all()); schema_ok=bool(out.columns.tolist()==champion.columns.tolist() and out.user_id.astype(str).tolist()==test.user_id.astype(str).tolist())
test_actions=int(sel.sum()); expected_cap=int(math.ceil(cfg.fixed_coverage*NTE))
action_cap_ok=bool(test_actions<=expected_cap)
CORRECTNESS_OK=bool(exact_locked and not membership.any() and rank5_same and multiset_ok and finite_ok and schema_ok and action_cap_ok)

# Primary fails closed only for correctness or competition-evidence RED/BORDERLINE status. Candidate is always retained for post-run audit.
if CORRECTNESS_OK and RECOMMEND_SUBMIT:
    primary_path.write_bytes(candidate_path.read_bytes())
else:
    primary_path.write_bytes(CHAMPION_BYTES)
    assert hashlib.sha256(primary_path.read_bytes()).hexdigest()==CHAMPION_SHA256

# Test behavior ledger.
test_ledger=pd.DataFrame({"user_id":test.user_id.astype(str),"eligible_s0":TEST_ELIGIBLE,"selected":sel,
                          "candidate":np.array(MODULES)[cand],"incumbent":np.array(MODULES)[inc],"pred_utility":score,
                          "champion_top1":np.array(MODULES)[champ_order[:,0]]})
test_ledger.to_csv(OUT/"v129_test_action_ledger.csv",index=False)

pkg={"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,"scipy":scipy.__version__,"sklearn":__import__('sklearn').__version__,"lightgbm":lgb.__version__}
json.dump(pkg,open(OUT/"v129_package_versions.json","w"),indent=2)
report={
    "version":"V129","status":STATUS,"recommend_submit":RECOMMEND_SUBMIT,"correctness_ok":CORRECTNESS_OK,
    "fixed_coverage":cfg.fixed_coverage,"primary_gain":float(primary["policy"]["gain"]),"benchmark_mix_gain":float(primary["mix_gain"]),
    "seen_gain":float(primary["seen_gain"]),"unseen_gain":float(primary["unseen_gain"]),"positive_random_folds":primary_pos,
    "primary_ci_user":primary["policy"]["ci_user"],"primary_ci_cluster":primary["policy"]["ci_cluster"],
    "severe_stress":severe_stress,"test_actions":test_actions,"test_action_rate":float(test_actions/NTE),
    "protected_exact":exact_locked,"top5_membership_unchanged":bool(not membership.any()),"rank5_unchanged":rank5_same,"score_multiset_preserved":multiset_ok,
    "champion_sha256":CHAMPION_SHA256,"candidate_sha256":hashlib.sha256(candidate_path.read_bytes()).hexdigest(),"primary_sha256":hashlib.sha256(primary_path.read_bytes()).hexdigest(),
    "submission_instruction":("FIRST-SLOT CANDIDATE: submit submission_v129_primary.csv after artifact audit." if (CORRECTNESS_OK and RECOMMEND_SUBMIT) else
                              "DO NOT AUTO-SUBMIT: primary failed closed to champion; inspect candidate_v129_DIAGNOSTIC.csv and diagnostics.")
}
json.dump(report,open(OUT/"v129_decision_report.json","w"),indent=2)
print(json.dumps(report,indent=2)); print("Outputs:",OUT,"elapsed final",round(time.time()-t0,1),"s")


{
  "version": "V129",
  "status": "GREEN",
  "recommend_submit": true,
  "correctness_ok": true,
  "fixed_coverage": 0.05,
  "primary_gain": 0.0011301930556073786,
  "benchmark_mix_gain": 0.0011703919159089214,
  "seen_gain": 0.0011925042934738958,
  "unseen_gain": 0.0010723283284468615,
  "positive_random_folds": 5,
  "primary_ci_user": [
    0.00036428423765464687,
    0.0018839645147905682
  ],
  "primary_ci_cluster": [
    0.0003157520465789268,
    0.0019904358119389163
  ],
  "severe_stress": false,
  "test_actions": 50,
  "test_action_rate": 0.05,
  "protected_exact": true,
  "top5_membership_unchanged": true,
  "rank5_unchanged": true,
  "score_multiset_preserved": true,
  "champion_sha256": "443785fb534df9820ce57bfe450a2768881625bf7ae37aa849dd0d9f2afc957d",
  "candidate_sha256": "34f334429c2113ba5b96283a7974fad472068ea08047ba3275fc44057e3a95df",
  "primary_sha256": "34f334429c2113ba5b96283a7974fad472068ea08047ba3275fc44057e3a95df",
  "submission_instruction": "FIRST-SLOT CAND

## Expected output artifacts

The full run writes dense caches plus:

- `v129_primary_fold_metrics.csv`
- `v129_primary_coverage_diagnostic.csv`
- `v129_primary_action_ledger.csv`
- `v129_primary_oof.npz`
- `v129_world_metrics.csv` and secondary fold diagnostics
- `v129_all_world_coverage_diagnostic.csv`
- `v129_action_feature_names.json`
- `v129_test_action_ledger.csv`
- `candidate_v129_DIAGNOSTIC.csv`
- `submission_v129_primary.csv`
- `v129_decision_report.json`
- `v129_package_versions.json`

`submission_v129_primary.csv` contains the V129 candidate only for `GREEN/YELLOW` evidence with all hard correctness invariants passing. Otherwise it is byte-identical to the protected `0.66028` champion; the diagnostic candidate remains available for audit.